<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/%EB%B2%A1%ED%84%B0%EC%8A%A4%ED%86%A0%EC%96%B4_Chroma%2C_FAISS_%2B_BM25_%EC%97%B0%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [74]:
!pip install langchain chromadb transformers sentence-transformers pypdf langchain-community langchain-huggingface faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.8 MB/s eta 0:00:00


In [2]:
!wget https://wdr.ubion.co.kr/wowpass/img/event/gsat_170823/gsat_170823.pdf

--2026-03-07 07:10:33--  https://wdr.ubion.co.kr/wowpass/img/event/gsat_170823/gsat_170823.pdf
Resolving wdr.ubion.co.kr (wdr.ubion.co.kr)... 61.100.182.43
Connecting to wdr.ubion.co.kr (wdr.ubion.co.kr)|61.100.182.43|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1253369 (1.2M) [application/pdf]
Saving to: ‘gsat_170823.pdf’

gsat_170823.pdf     100%[===================>]   1.19M   927KB/s    in 1.3s    

2026-03-07 07:10:36 (927 KB/s) - ‘gsat_170823.pdf’ saved [1253369/1253369]



#데이터 불러오기

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
## pdf 파일로드 하고 쪼개기
loader = PyPDFLoader('https://wdr.ubion.co.kr/wowpass/img/event/gsat_170823/gsat_170823.pdf')
pages = loader.load_and_split()
print(len(pages))

## chunk로 쪼개기
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splited_docs = text_splitter.split_documents(pages)
print(len(splited_docs))

27
69


#1. Chroma DB

In [5]:
model_huggingface = HuggingFaceEmbeddings(model_name='BAAI/bge-m3')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [8]:
chroma_db = Chroma.from_documents(splited_docs, model_huggingface, persist_directory = './samsumg.db')
print('문서의 수 : ', chroma_db._collection.count())

문서의 수 :  207


문서추가

In [23]:
from langchain_core.documents import Document

new_doc = Document(
    page_content = "나는 가렌이다! 전진! 가붕가붕!",
    metadata = {
        'source' : 'leageoflegends.com',
        'page' : 10000
    }
)

In [28]:
chroma_db.add_documents([new_doc], id=['99'])

['ffb375a1-8d61-4311-86af-51152029e875']

문서삭제

In [70]:
#삭제하고자 하는 문서 검색
result = chroma_db.similarity_search('가붕가붕')
delete_text = result[0].page_content

#삭제하고자 하는 문서의 id 반환
for i, v in enumerate(chroma_db._collection.get()['documents']):
  if delete_text in v:
    print('index : ', i)

delete_id = chroma_db._collection.get()['ids'][207]
print('id : ', delete_id)

#삭제
print('삭제 전 : ', chroma_db._collection.count())
chroma_db.delete(ids = [delete_id])
print('삭제 후 : ', chroma_db._collection.count())


index :  207
id :  ffb375a1-8d61-4311-86af-51152029e875
삭제 전 :  208
삭제 후 :  207


#2. FAISS

In [75]:
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(splited_docs, model_huggingface)

In [82]:
faiss_db.docstore._dict

{'efc444ba-5782-4036-88d1-f8ac9c2176cb': Document(id='efc444ba-5782-4036-88d1-f8ac9c2176cb', metadata={'producer': 'itext-paulo-155 (itextpdf.sf.net-lowagie.com)', 'creator': 'nPDF (pdftk 1.41)', 'creationdate': '2017-08-16T00:21:02-08:00', 'moddate': '2017-08-16T00:21:02-08:00', 'source': 'https://wdr.ubion.co.kr/wowpass/img/event/gsat_170823/gsat_170823.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1'}, page_content='2\n01 삼성전자 기업분석\n(Samsung Electronics Co., Ltd)\nⅠ 기업 일반 \n1  기업개요\n1) 기업소개 \n본사주소 경기도 수원시 영통구 삼성로 129(매탄동 416)\n사업분야 삼성그룹의 대표 기업으로 휴대폰, 정보통신기기, 반도체, TV 등을 생산 판매하는 제조업체\n홈페이지 www.samsung.com/sec 구분 전기전자 대기업  \n설립일 1961년 07월 01일 대표이사 권오현 \n총자산1) 244조 매출액2) 200조\n임직원수 95,374명 \n∙ 1975년 1월 주식시장 상장\n∙ 1984년 2월 삼성전자공업주식회사->삼성전자주식회사로 사명 변경 \n∙ CE(Consumer Electronics), IM(Information technology & Mobile communications), DS(Device Solutions) \n3개의 부문으로 나누어 독립 경영.\n부문 제품\nCE TV, 모니터, 냉장고, 세탁기, 에어컨, 프린터, 의료기기 등'),
 '3bf7cd23-3ba3-4389-9dee-90ea463b0d73': Document(id='3bf7cd2

Document로 추가

In [84]:
from langchain_core.documents import Document

new_doc = Document(page_content = "가붕가붕 가렌이다!", metadata = {'sourde' : 'leage of legends'})

print('추가 전 : ', faiss_db.index.ntotal)
faiss_db.add_documents([new_doc])
print('추가 후 : ', faiss_db.index.ntotal)

['b030bb97-7a65-4d4f-9f44-6f1fa25719e8']

DB Merge

In [88]:
faiss_db.index.ntotal

70

In [89]:
texts = ['최한나 이쁘다','최한나 결혼하자']
faiss_hanna_db = FAISS.from_texts(texts, model_huggingface)

In [90]:
faiss_db.merge_from(faiss_hanna_db)

In [91]:
faiss_db.index.ntotal

72

문서 삭제

In [100]:
documents = faiss_db.similarity_search('최한나')
ids = [documents[0].id, documents[1].id]

faiss_db.delete(ids)

True

In [101]:
faiss_db.similarity_search('최한나')

[Document(id='cd95dc54-bb16-49b2-b3d6-719a8bc43d4c', metadata={'producer': 'itext-paulo-155 (itextpdf.sf.net-lowagie.com)', 'creator': 'nPDF (pdftk 1.41)', 'creationdate': '2017-08-16T00:21:02-08:00', 'moddate': '2017-08-16T00:21:02-08:00', 'source': 'https://wdr.ubion.co.kr/wowpass/img/event/gsat_170823/gsat_170823.pdf', 'total_pages': 27, 'page': 21, 'page_label': '22'}, page_content='연구개발직 전기전자 (H/W), 재료/금속, 화학/화공, 물리, 기계 기흥 /화성/온양\n설비엔지니어직 전기전자(H/W), 재료/금속, 화학/화공, 물리, 기계 기흥 /화성/온양\nTest&Pack\nage 센터\n연구개발직 전기전자 (H/W), 재료/금속, 화학/화공, 물리, 기계 온양\n설비엔지니어직 전기전자(H/W), 재료/금속, 화학/화공, 물리, 기계 온양\nLED사업팀 연구개발직 전기전자 (H/W), 재료/금속, 화학/화공, 물리, 기계 기흥 /화성\n기흥/화성\n단지총괄\n연구개발직 전기전자 (H/W), 기계, 산공 기흥 /화성/온양\n설비엔지니어직 전기전자(H/W), 재료/금속, 화학/화공, 물리, 기계 기흥 /화성/온양\n생산기술\n연구소 연구개발직 전기전자 (H/W), 물리, 기계 화성\n부문공통\n소프트웨어직 전기전자(S/W), 전산/컴퓨터, 산공, 수학, 물리, 통계 기흥/화성/수원/\n온양\n영업마케팅직 전공무관 기흥 /화성\n경영지원직(재무) 상경(부전공 포함) 기흥/화성/온양'),
 Document(id='30f094f4-a235-4850-9144-492acb9d45ec', metadata={'producer': 'itext-paulo-155 (it